In [ ]:
# %%
#!/usr/bin/env python

"""
MultiGrate + Linear Regression Pipeline
Target: Continuous protein levels (CD45RA)
"""

from __future__ import annotations
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import multigrate as mtg
import scvi
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import joblib

In [ ]:
# -----------------------------
# Data Loaders (Aligned with JACA/MOFA structure)
# -----------------------------

def load_tea_seq(target_name: str = None, base_dir: str = "../data"):
    """
    Updated loader to read directly from base_dir.
    If target_name is provided, it loads the adt_minus file for that protein.
    """
    base = Path(base_dir)
    
    print(f"[Loading] RNA and ATAC from {base}")
    rna = ad.read_h5ad(base / "rna.h5ad")
    atac = ad.read_h5ad(base / "atac.h5ad")
    
    if target_name:
        adt_path = base / f"adt_minus_{target_name}.h5ad"
        print(f"[Loading] ADT-minus for {target_name}: {adt_path.name}")
        adt = ad.read_h5ad(adt_path)
    else:
        # Fallback to standard adt if no specific target is requested
        print(f"[Loading] Standard ADT from {base / 'adt.h5ad'}")
        adt = ad.read_h5ad(base / "adt.h5ad")
        
    return rna, atac, adt

def load_train_indices(
    splits_dir: str,
    split_tag: str,
    n_cells: int,
):
    """
    Load train indices.
    DEFAULT ASSUMPTION: indices are 0-based.

    Converts to 0-based *only* if the file is unambiguously 1-based
    (min==1 and max==n_cells).
    """
    path = Path(splits_dir) / f"{split_tag}_train_idx.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")

    df = pd.read_csv(path)
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        raise ValueError(f"No numeric column found in {path}")

    idx = df[num_cols[0]].to_numpy()

    # ---- require integer-valued ----
    if np.isnan(idx).any():
        raise ValueError(f"NaNs found in {path}")
    idx_int = idx.astype(np.int64)
    if not np.allclose(idx, idx_int):
        raise ValueError(f"Non-integer indices found in {path}")
    idx = idx_int

    mn, mx = int(idx.min()), int(idx.max())

    # ---- sentinel-based conversion only ----
    if mn == 1 and mx == n_cells:
        idx0 = idx - 1
        mode = "1-based->0-based"
    else:
        idx0 = idx
        mode = "assumed-0-based"

    # ---- strict bounds check (NO clipping) ----
    if (idx0 < 0).any() or (idx0 >= n_cells).any():
        raise ValueError(
            f"Out-of-range indices in {path}: "
            f"min={idx0.min()}, max={idx0.max()}, n_cells={n_cells}, mode={mode}"
        )

    # Optional debug line (comment out once stable)
    # print(f"[load_train_indices] {split_tag}: {mode}, n={len(idx0)}")

    return idx0

def load_regression_target(target_protein: str, base_dir: str, cell_names: pd.Index):
    """
    Loads the protein level CSV and aligns it with the training cell names.
    """
    path = Path(base_dir) / "response" / f"{target_protein}.csv"
    df_y = pd.read_csv(path, index_col=0)
    # Reindex ensures we have the same order as our AnnData
    return df_y.loc[cell_names].iloc[:, 0].values.astype(np.float32)


In [ ]:
# -----------------------------
# MultiGrate Setup
# -----------------------------

def build_multigrate_adata(rna, atac, adt):
    # Ensure float32 for non-RNA
    atac.X = atac.layers["norm"].astype(np.float32)
    adt.X = adt.layers["norm"].astype(np.float32)
    
    adatas = [[rna], [atac], [adt]]
    # RNA uses .X (counts), Others use 'norm' layer
    adata = mtg.data.organize_multimodal_anndatas(
        adatas=adatas,
        layers=[["norm"], ["norm"], ["norm"]]
    )
    return adata, rna.shape[1]


In [ ]:
# -----------------------------
# Main Training Function
# -----------------------------

def train_multigrate_regression_ols(
    target_protein: str = "CD45RA",
    split_tag: str = "tea_split3_all_celltypes",
    base_dir: str = "../data",
    splits_dir: str = "../splits",
    model_dir: str = "../models",
    z_dim: int = 70,
    max_epochs: int = 200
):
    scvi.settings.seed = 0
    model_root = Path(model_dir) / f"{split_tag}_multigrate_ols"
    model_root.mkdir(parents=True, exist_ok=True)

    # 1. Load and Subset
    print(f"[Step 1] Loading data...")
    rna, atac, adt = load_tea_seq(target_name=target_protein, base_dir=base_dir)
    train_idx = load_train_indices(splits_dir, split_tag, rna.n_obs)
    train_names = rna.obs_names[train_idx]

    rna_tr, atac_tr, adt_tr = rna[train_names].copy(), atac[train_names].copy(), adt[train_names].copy()

    # 2. MultiGrate Embedding
    print(f"[Step 2] Training MultiGrate (z_dim={z_dim})...")
    adata_tr, rna_end = build_multigrate_adata(rna_tr, atac_tr, adt_tr)
    
    mtg.model.MultiVAE.setup_anndata(adata_tr, rna_indices_end=rna_end)
    vae = mtg.model.MultiVAE(adata_tr, losses=["mse", "mse", "mse"], z_dim=z_dim)
    vae.train(max_epochs=max_epochs)
    
    # Extract Latent Space
    vae.get_model_output()
    Z_train = adata_tr.obsm["X_multigrate"]

    # 3. Regression Target
    print(f"[Step 3] Loading {target_protein} target values...")
    y_train = load_regression_target(target_protein, base_dir, train_names)

    # 4. Simple Linear Regression (OLS)
    print(f"[Step 4] Fitting OLS Linear Regression...")
    reg = LinearRegression()
    reg.fit(Z_train, y_train)
    
    # Calculate Metrics
    y_pred = reg.predict(Z_train)
    r2 = r2_score(y_train, y_pred)
    pearson_corr = np.corrcoef(y_train, y_pred)[0, 1]
    
    print(f"\n--- Training Results ---")
    print(f"Target: {target_protein}")
    print(f"R-squared: {r2:.4f}")
    print(f"Pearson Correlation: {pearson_corr:.4f}")

    # 5. Save
    vae.save(str(model_root / "multivae_model"), overwrite=True)
    joblib.dump(reg, model_root / f"ols_regressor_{target_protein}.pkl")
    
    # Optional Visualization
    obs_key = f"{target_protein}_target"
    adata_tr.obs[obs_key] = y_train
    print(f"[Step 6] Computing UMAP and plotting {obs_key}...")
    sc.pp.neighbors(adata_tr, use_rep="X_multigrate")
    sc.tl.umap(adata_tr)
    
    # Plot using the unique obs_key
    sc.pl.umap(
        adata_tr, 
        color=obs_key, 
        title=f"MultiGrate Embedding: {target_protein} Target Values"
    )

# %%
if __name__ == "__main__":
    train_multigrate_regression_ols()